# LSTM Word2Vec Baseline

Frozen Word2Vec embeddings with a unidirectional LSTM classifier.

In [ ]:
from pathlib import Path
import os
import sys
from IPython.display import Image, display

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

CONFIG_PATH = PROJECT_ROOT / "configs" / "experiment.yaml"
FIGURE_DIR = PROJECT_ROOT / "reports" / "figure" / "02_training"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

def display_saved_figures(paths):
    if isinstance(paths, dict):
        paths = paths.values()
    elif isinstance(paths, (str, Path)):
        paths = [paths]

    for path in paths:
        display(Image(filename=str(path)))

CONFIG_PATH

In [ ]:
from src.trainer import train_baseline

history = train_baseline(str(CONFIG_PATH))
history

In [ ]:
from src.utils import load_config, plot_training_curves

config = load_config(CONFIG_PATH)
curve_paths = plot_training_curves(
    history,
    model_name="lstm_w2vec",
    max_features=config["features"]["word2vec"]["vector_size"],
    dropout=config["models"]["lstm"].get("dropout", 0.0),
    output_dir=FIGURE_DIR,
)
display_saved_figures(curve_paths)
curve_paths

In [ ]:
import torch
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

from src.trainer import build_model, prepare_lstm_dataloaders, resolve_device
from src.utils import plot_confusion_matrix, plot_metrics_bar

config = load_config(CONFIG_PATH)
device = resolve_device(config["training"]["device"])
_, val_loader, artifacts = prepare_lstm_dataloaders(config)
model = build_model(config, embedding_matrix=artifacts["embedding_matrix"]).to(device)

checkpoint = torch.load(PROJECT_ROOT / "checkpoints" / "lstm_w2vec_best_model.pt", map_location=device)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

y_true = []
y_pred = []
def unpack_lstm_batch(batch):
    if len(batch) == 3:
        return batch

    x_batch, y_batch = batch
    lengths = (x_batch != 0).sum(dim=1).clamp(min=1)
    return x_batch, lengths, y_batch

with torch.no_grad():
    for batch in val_loader:
        x_batch, lengths, y_batch = unpack_lstm_batch(batch)
        logits = model(x_batch.to(device), lengths.to(device))
        preds = torch.argmax(logits, dim=1)
        y_pred.extend(preds.cpu().numpy())
        y_true.extend(y_batch.cpu().numpy())

precision, recall, f1, _ = precision_recall_fscore_support(y_true, y_pred, average="macro", zero_division=0)
metrics = {
    "accuracy": accuracy_score(y_true, y_pred),
    "precision": precision,
    "recall": recall,
    "f1": f1,
}

cm_path = plot_confusion_matrix(y_true, y_pred, ["Negative", "Neutral", "Positive"], "lstm_w2vec", FIGURE_DIR)
metrics_path = plot_metrics_bar(metrics, "lstm_w2vec", "metrics", FIGURE_DIR)
display_saved_figures([cm_path, metrics_path])
metrics, cm_path, metrics_path